# Embeddings Explorer

**Goal:** Visualize high-dimensional embeddings in 2D and 3D space. See how similar concepts cluster together, explore similarity neighborhoods, and compare embedding models — building the intuition for why vector search works.

**No API keys required** — this notebook uses open-source models that run locally.

---

## What You'll Learn

1. How embeddings organize text by meaning in high-dimensional space
2. Why similar concepts cluster together (and how to see it)
3. How to explore similarity neighborhoods around any query
4. Whether embeddings can capture semantic relationships (the analogy test)
5. How different embedding models compare

## Setup

In [ ]:
# Uncomment to install dependencies
# !pip install sentence-transformers scikit-learn matplotlib plotly numpy

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✓ Model loaded: all-MiniLM-L6-v2 ({model.get_sentence_embedding_dimension()} dimensions)")
print(f"\nSetup complete!")

## Step 1: Build a Diverse Corpus

We need texts across multiple domains and semantic distances to see how embeddings organize them. We'll use 30 sentences across 6 business categories from the same consulting firm context.

In [ ]:
corpus = {
    'Financial': [
        "Q3 revenue reached $42.3 million, up 12% year-over-year.",
        "Gross margin improved to 38.5% from 36.2% in the previous quarter.",
        "Operating expenses were held flat at $12.1 million despite growth.",
        "The backlog stands at $127 million providing 9 months visibility.",
        "New bookings totaled $38 million with a book-to-bill of 0.90.",
    ],
    'HR & People': [
        "Total headcount reached 852 employees, a net increase of 28.",
        "Voluntary attrition fell to 8% annualized from 11% last year.",
        "The firm hired 45 new consultants with an 87% acceptance rate.",
        "Employee engagement score held steady at 4.1 out of 5.0.",
        "The offshore team in India grew to 45 members.",
    ],
    'Client': [
        "Client satisfaction averaged 4.7 out of 5.0, a firm record.",
        "The firm onboarded 12 new clients bringing total to 94.",
        "Net Promoter Score improved to 62, up from 58 last quarter.",
        "Eight existing clients expanded their engagements.",
        "Client retention rate remains strong at 92%.",
    ],
    'Technology': [
        "Azure cloud migration is 85% complete across all units.",
        "The GenAI dashboard processes 340 queries per day.",
        "AI-related revenue reached $6.8 million, up 113% YoY.",
        "FedRAMP authorization maintained with zero security incidents.",
        "Cloud costs are tracking 15% below budget.",
    ],
    'Legal & Compliance': [
        "SOC 2 Type II audit completed with no findings.",
        "FedRAMP compliance maintained for all cloud services.",
        "The MCP security framework pilot shows promising results.",
        "Data residency requirements met for all government workloads.",
        "Zero security incidents reported across all platforms.",
    ],
    'Strategy': [
        "Investing in grid modernization and clean energy advisory.",
        "Shifting toward smaller deals that build relationships for follow-on.",
        "The DoD AI modernization contract will provide revenue through 2027.",
        "AI-focused engagements represent a growing share of revenue.",
        "Expansion revenue contributed 34% of total Q3 revenue.",
    ],
}

# Flatten for embedding
all_texts = []
all_categories = []
for category, sentences in corpus.items():
    all_texts.extend(sentences)
    all_categories.extend([category] * len(sentences))

# Embed everything
all_embeddings = model.encode(all_texts)

print(f"✓ Embedded {len(all_texts)} sentences across {len(corpus)} categories")
print(f"  Each sentence → {all_embeddings.shape[1]}-dimensional vector")
for cat, sents in corpus.items():
    print(f"  • {cat}: {len(sents)} sentences")

## Step 2: 2D Projection with t-SNE

384 dimensions are impossible to visualize. **t-SNE** (t-distributed Stochastic Neighbor Embedding) projects them to 2D while preserving relative distances — nearby points in 384D stay nearby in 2D.

If embeddings capture meaning, we should see business categories form distinct clusters.

In [ ]:
# Project to 2D
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=8, n_iter=1000)
embeddings_2d = tsne_2d.fit_transform(all_embeddings)

# Color map
colors = {
    'Financial': '#2E86C1',
    'HR & People': '#1E8449',
    'Client': '#D35400',
    'Technology': '#8E44AD',
    'Legal & Compliance': '#C0392B',
    'Strategy': '#F39C12',
}

fig, ax = plt.subplots(figsize=(14, 10))

for category in colors:
    mask = [c == category for c in all_categories]
    indices = [i for i, m in enumerate(mask) if m]
    ax.scatter(
        embeddings_2d[indices, 0], embeddings_2d[indices, 1],
        c=colors[category], label=category, s=150, alpha=0.8,
        edgecolors='white', linewidth=1.5,
    )

# Add labels for selected points
for i, text in enumerate(all_texts):
    short = text[:45] + '...' if len(text) > 45 else text
    ax.annotate(short, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
               fontsize=7, ha='center', va='bottom',
               xytext=(0, 8), textcoords='offset points', alpha=0.7)

ax.legend(fontsize=11, loc='best')
ax.set_title('Embedding Space: 30 Business Sentences in 2D (t-SNE)', fontsize=14, fontweight='bold')
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("🎯 Key Insight: Sentences about similar topics cluster together.")
print("   This is the foundation of semantic search — find the nearest cluster.")

## Step 3: Interactive 3D Visualization

2D loses some spatial relationships. Let's project to 3D for an interactive view you can rotate and explore.

In [ ]:
# Project to 3D
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=8, n_iter=1000)
embeddings_3d = tsne_3d.fit_transform(all_embeddings)

# Build interactive 3D scatter
fig = go.Figure()

for category in colors:
    mask = [c == category for c in all_categories]
    indices = [i for i, m in enumerate(mask) if m]
    texts_for_hover = [all_texts[i][:60] + '...' for i in indices]
    
    fig.add_trace(go.Scatter3d(
        x=embeddings_3d[indices, 0],
        y=embeddings_3d[indices, 1],
        z=embeddings_3d[indices, 2],
        mode='markers',
        name=category,
        marker=dict(size=8, color=colors[category], opacity=0.8),
        text=texts_for_hover,
        hoverinfo='text+name',
    ))

fig.update_layout(
    title='Embedding Space: Interactive 3D View',
    scene=dict(xaxis_title='Dim 1', yaxis_title='Dim 2', zaxis_title='Dim 3'),
    width=900, height=700,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()
print("💡 Rotate the 3D plot to see cluster separation from different angles.")
print("   Hover over points to see the full text.")

## Step 4: Explore Similarity Neighborhoods

Pick any sentence and see its nearest and farthest neighbors. This shows exactly what the vector database "sees" when you search.

In [ ]:
def explore_neighborhood(query_text, all_texts, all_embeddings, model, top_k=5):
    """Find nearest and farthest neighbors for a query."""
    query_emb = model.encode([query_text])
    sims = cosine_similarity(query_emb, all_embeddings)[0]
    
    sorted_idx = np.argsort(sims)[::-1]
    nearest = sorted_idx[:top_k]
    farthest = sorted_idx[-top_k:][::-1]
    
    print(f"🔎 Query: \"{query_text}\"")
    print(f"\n  {'NEAREST NEIGHBORS':^60}")
    print(f"  {'-'*60}")
    for i, idx in enumerate(nearest):
        cat = all_categories[idx]
        print(f"  {i+1}. (sim: {sims[idx]:.4f}) [{cat}] {all_texts[idx][:65]}")
    
    print(f"\n  {'FARTHEST NEIGHBORS':^60}")
    print(f"  {'-'*60}")
    for i, idx in enumerate(farthest):
        cat = all_categories[idx]
        print(f"  {i+1}. (sim: {sims[idx]:.4f}) [{cat}] {all_texts[idx][:65]}")
    
    return nearest, farthest, sims


# Explore a financial query
nearest, farthest, sims = explore_neighborhood(
    "How is the company performing financially?",
    all_texts, all_embeddings, model
)

print(f"\n💡 Notice: nearest neighbors are all financial, farthest are HR/legal.")
print(f"   This is exactly how RAG retrieval filters relevant context.")

In [ ]:
# Visualize the neighborhood on the 2D plot
fig, ax = plt.subplots(figsize=(14, 10))

# Plot all points (faded)
for category in colors:
    mask = [c == category for c in all_categories]
    indices = [i for i, m in enumerate(mask) if m]
    ax.scatter(embeddings_2d[indices, 0], embeddings_2d[indices, 1],
              c=colors[category], s=80, alpha=0.2, label=category)

# Highlight nearest (green rings)
for idx in nearest:
    ax.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
             s=200, facecolors='none', edgecolors='#27AE60', linewidth=2.5)
    ax.annotate(all_texts[idx][:35]+'...', (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
               fontsize=8, fontweight='bold', color='#27AE60',
               xytext=(10, 10), textcoords='offset points')

# Highlight farthest (red rings)
for idx in farthest:
    ax.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
             s=200, facecolors='none', edgecolors='#E74C3C', linewidth=2.5)

# Plot query point
query_emb_2d = tsne_2d.fit_transform(
    np.vstack([all_embeddings, model.encode(["How is the company performing financially?"])])
)[-1]

ax.scatter(query_emb_2d[0], query_emb_2d[1], c='black', s=300, marker='*', zorder=5)
ax.annotate('QUERY', (query_emb_2d[0], query_emb_2d[1]),
           fontsize=10, fontweight='bold', xytext=(0, 15), textcoords='offset points', ha='center')

ax.legend(fontsize=10)
ax.set_title('Similarity Neighborhood: Green = Nearest, Red = Farthest', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Step 5: The Analogy Test

Famous finding from word2vec: vector arithmetic can capture relationships. Does `king - man + woman ≈ queen` work with sentence embeddings? Let's test with business concepts.

In [ ]:
def analogy_test(a, b, c, all_texts, all_embeddings, model):
    """
    Test: A is to B as C is to ???
    Compute: embedding(B) - embedding(A) + embedding(C)
    Find the closest match in the corpus.
    """
    emb_a = model.encode([a])[0]
    emb_b = model.encode([b])[0]
    emb_c = model.encode([c])[0]
    
    # Vector arithmetic
    target = emb_b - emb_a + emb_c
    target = target.reshape(1, -1)
    
    # Find closest in corpus
    sims = cosine_similarity(target, all_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:3]
    
    print(f"  \"{a}\" → \"{b}\"")
    print(f"  \"{c}\" → ???\n")
    print(f"  Top matches:")
    for i, idx in enumerate(top_idx):
        print(f"    {i+1}. (sim: {sims[idx]:.4f}) {all_texts[idx][:70]}")
    print()


print("Analogy Test: Does Vector Arithmetic Work?")
print("=" * 60)

print("\n📝 Test 1: Revenue is to financial as headcount is to...")
analogy_test(
    "Q3 revenue reached $42.3 million",
    "Gross margin improved to 38.5%",
    "Total headcount reached 852 employees",
    all_texts, all_embeddings, model
)

print("📝 Test 2: Hiring is to people as migration is to...")
analogy_test(
    "The firm hired 45 new consultants",
    "Voluntary attrition fell to 8%",
    "Azure cloud migration is 85% complete",
    all_texts, all_embeddings, model
)

print("💡 Analogy quality depends on the embedding model and corpus size.")
print("   Sentence-level embeddings capture relationships less cleanly")
print("   than word-level embeddings, but the direction is correct.")

## Step 6: Model Comparison

Different embedding models capture meaning differently. Let's compare `all-MiniLM-L6-v2` (fast, 384d) with `all-mpnet-base-v2` (slower, 768d, higher quality) on the same corpus.

In [ ]:
# Load second model
model_b = SentenceTransformer('all-mpnet-base-v2')
print(f"✓ Model B loaded: all-mpnet-base-v2 ({model_b.get_sentence_embedding_dimension()} dimensions)")

# Embed with both models
emb_a = model.encode(all_texts)
emb_b = model_b.encode(all_texts)

sim_a = cosine_similarity(emb_a)
sim_b = cosine_similarity(emb_b)

# Side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

cat_labels = [f"{all_categories[i][:3]}:{all_texts[i][:20]}" for i in range(len(all_texts))]

for ax, sim, title in zip(axes, [sim_a, sim_b],
                           ['all-MiniLM-L6-v2 (384d)', 'all-mpnet-base-v2 (768d)']):
    im = ax.imshow(sim, cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title(title, fontsize=13, fontweight='bold')
    
    # Add category separators
    for i in range(0, len(all_texts), 5):
        ax.axhline(y=i-0.5, color='black', linewidth=0.5)
        ax.axvline(x=i-0.5, color='black', linewidth=0.5)
    
    # Category labels on axes
    cat_names = list(corpus.keys())
    ax.set_xticks([2, 7, 12, 17, 22, 27])
    ax.set_xticklabels(cat_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticks([2, 7, 12, 17, 22, 27])
    ax.set_yticklabels(cat_names, fontsize=9)

plt.colorbar(im, ax=axes, label='Cosine Similarity', shrink=0.8)
plt.suptitle('Similarity Matrices: Two Embedding Models Compared', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantitative comparison
print("\n📊 Quantitative Comparison:")
print(f"{'Metric':<35} {'MiniLM (384d)':>15} {'MPNet (768d)':>15}")
print(f"{'-'*35} {'-'*15} {'-'*15}")

# Average within-category similarity
for cat in corpus:
    mask = [c == cat for c in all_categories]
    idx = [i for i, m in enumerate(mask) if m]
    within_a = np.mean([sim_a[i][j] for i in idx for j in idx if i != j])
    within_b = np.mean([sim_b[i][j] for i in idx for j in idx if i != j])
    print(f"  Within {cat:<25} {within_a:>15.3f} {within_b:>15.3f}")

print(f"\n💡 Higher within-category similarity = better semantic grouping.")
print(f"   MPNet generally produces tighter clusters at the cost of 2x dimensions.")

## Key Takeaways

1. **Embeddings organize text by meaning.** Financial sentences cluster with financial sentences, HR with HR — automatically, without any labels or rules. This is what makes semantic search possible.

2. **Dimensionality reduction (t-SNE) reveals structure.** Even though embeddings live in 384+ dimensions, the semantic relationships are visible when projected to 2D/3D. Clusters, neighborhoods, and distances are real.

3. **Nearest neighbors = search results.** When a vector database returns "similar documents," it's finding the nearest neighbors in this space. The quality of the embedding model directly determines the quality of search.

4. **Vector arithmetic partially works.** Sentence-level embeddings capture directional relationships (financial → financial metric ≈ HR → HR metric), but less cleanly than word-level embeddings. Don't rely on it for production.

5. **Model choice is a trade-off.** More dimensions = better separation but higher storage/compute. For prototyping: MiniLM (384d, fast). For production: MPNet (768d) or OpenAI embedding-3-large (3072d).

---

### Next Steps
- **[01-semantic-search-visualized.ipynb](01-semantic-search-visualized.ipynb)** — See semantic search in action with queries
- **[03-chunking-strategies.ipynb](03-chunking-strategies.ipynb)** — How chunking affects what gets embedded
- **[RAG Deep Dive](../docs/04-rag-deep-dive.md)** — Full engineering guide to the RAG pipeline